In [7]:
import sqlite3
import pandas as pd
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [8]:
project_root = '/Users/nicoleotero/Desktop/GLP1 research project/glp1-market-impact-analysis'
raw_data_path = os.path.join(project_root, 'data', 'raw', 'close_prices_raw.csv')
db_path = os.path.join(project_root, 'data', 'processed', 'glp1_analysis.db')

close_prices = pd.read_csv(raw_data_path, index_col=0, parse_dates=True)

print("Data loaded successfully")
print(f"Shape: {close_prices.shape}")
close_prices.head()

Data loaded successfully
Shape: (1884, 11)


,DXCM,ISRG,KHC,LLY,MCD,NSRGY,NVO,PEP,PLNT,PODD,YUM
Date,,,,,,,,,,,
2019-01-02,28.795000,155.343338,29.806629,104.162521,147.795364,65.293228,19.790524,86.693657,53.580002,73.430000,79.630615
2019-01-03,28.065001,150.080002,29.786001,100.925621,146.821594,66.954659,19.951250,85.884483,52.799999,72.440002,77.627632
2019-01-04,29.059999,157.226669,30.597525,103.963058,149.658951,68.078590,20.023157,87.645653,54.919998,77.150002,79.648010
2019-01-07,32.487499,159.479996,31.175234,104.525261,151.287521,67.361893,20.150049,86.891991,55.840000,74.209999,79.560913
2019-01-08,32.880001,160.996674,31.202742,105.486351,151.606522,67.573647,20.446119,87.724991,57.619999,74.150002,79.404175


In [9]:
# Create processed folder if needed
os.makedirs(os.path.join(project_root, 'data', 'processed'), exist_ok=True)

conn = sqlite3.connect(db_path)
print(f"Database created at: {db_path}")

sector_map = {
    'LLY':   'glp1_makers',
    'NVO':   'glp1_makers',
    'DXCM':  'diabetes_devices',
    'PODD':  'diabetes_devices',
    'KHC':   'food_beverage',
    'NSRGY': 'food_beverage',
    'PEP':   'food_beverage',
    'MCD':   'fast_food',
    'YUM':   'fast_food',
    'PLNT':  'fitness',
    'ISRG':  'surgical'
}

prices_long = close_prices.reset_index().melt(
    id_vars='Date',
    var_name='ticker',
    value_name='close_price'
)
prices_long['sector'] = prices_long['ticker'].map(sector_map)

prices_long.to_sql('stock_prices', conn, if_exists='replace', index=False)
print("stock_prices table created")

sectors_df = pd.DataFrame([{'ticker': k, 'sector': v} for k, v in sector_map.items()])
sectors_df.to_sql('sectors', conn, if_exists='replace', index=False)
print("sectors table created")

events = {
    'wegovy_approved':   '2021-06-04',
    'mounjaro_approved': '2022-05-13',
    'zepbound_approved': '2023-11-08'
}

events_df = pd.DataFrame([{'event': k, 'date': v} for k, v in events.items()])
events_df.to_sql('fda_events', conn, if_exists='replace', index=False)
print("fda_events table created")

# Add SPY
spy_path = os.path.join(project_root, 'data', 'raw', 'spy_benchmark.csv')
spy = pd.read_csv(spy_path, index_col=0, parse_dates=True)
spy.columns = ['close_price']
spy['ticker'] = 'SPY'
spy['sector'] = 'benchmark'
spy = spy.reset_index().rename(columns={'Date': 'Date'})
spy.to_sql('stock_prices', conn, if_exists='append', index=False)
print("SPY added")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(f"\nTables: {cursor.fetchall()}")

Database created at: /Users/nicoleotero/Desktop/GLP1 research project/glp1-market-impact-analysis/data/processed/glp1_analysis.db
stock_prices table created
sectors table created
fda_events table created
SPY added

Tables: [('stock_prices',), ('sectors',), ('fda_events',)]


In [10]:
processed_path = os.path.join(project_root, 'data', 'processed')
os.makedirs(processed_path, exist_ok=True)

# Query 1: Price summary
query1 = """
SELECT 
    ticker, sector,
    ROUND(AVG(close_price), 2) as avg_price,
    ROUND(MIN(close_price), 2) as min_price,
    ROUND(MAX(close_price), 2) as max_price
FROM stock_prices
GROUP BY ticker, sector
ORDER BY sector, ticker;
"""
result1 = pd.read_sql_query(query1, conn)
result1.to_csv(os.path.join(processed_path, 'price_summary.csv'), index=False)
print("=== QUERY 1: Price Summary ===")
print(result1.to_string(index=False))

# Query 2: Before vs after Wegovy
query2 = """
SELECT 
    ticker, sector,
    ROUND(AVG(CASE WHEN Date < '2021-06-04' THEN close_price END), 2) as avg_before_wegovy,
    ROUND(AVG(CASE WHEN Date >= '2021-06-04' THEN close_price END), 2) as avg_after_wegovy,
    ROUND(
        (AVG(CASE WHEN Date >= '2021-06-04' THEN close_price END) - 
         AVG(CASE WHEN Date < '2021-06-04' THEN close_price END)) /
         AVG(CASE WHEN Date < '2021-06-04' THEN close_price END) * 100
    , 2) as pct_change
FROM stock_prices
GROUP BY ticker, sector
ORDER BY pct_change DESC;
"""
result2 = pd.read_sql_query(query2, conn)
result2.to_csv(os.path.join(processed_path, 'before_after_wegovy.csv'), index=False)
print("\n=== QUERY 2: Before vs After Wegovy ===")
print(result2.to_string(index=False))

# Query 3: Relative to market
query3 = """
WITH spy_performance AS (
    SELECT 
        ROUND(
            (AVG(CASE WHEN Date >= '2021-06-04' THEN close_price END) - 
             AVG(CASE WHEN Date < '2021-06-04' THEN close_price END)) /
             AVG(CASE WHEN Date < '2021-06-04' THEN close_price END) * 100
        , 2) as spy_pct_change
    FROM stock_prices WHERE ticker = 'SPY'
),
stock_performance AS (
    SELECT ticker, sector,
        ROUND(
            (AVG(CASE WHEN Date >= '2021-06-04' THEN close_price END) - 
             AVG(CASE WHEN Date < '2021-06-04' THEN close_price END)) /
             AVG(CASE WHEN Date < '2021-06-04' THEN close_price END) * 100
        , 2) as stock_pct_change
    FROM stock_prices WHERE ticker != 'SPY'
    GROUP BY ticker, sector
)
SELECT s.ticker, s.sector, s.stock_pct_change,
    sp.spy_pct_change as market_pct_change,
    ROUND(s.stock_pct_change - sp.spy_pct_change, 2) as relative_to_market
FROM stock_performance s
CROSS JOIN spy_performance sp
ORDER BY relative_to_market DESC;
"""
result3 = pd.read_sql_query(query3, conn)
result3.to_csv(os.path.join(processed_path, 'relative_to_market.csv'), index=False)
print("\n=== QUERY 3: Relative to Market ===")
print(result3.to_string(index=False))

# Query 4: Event volatility
query4 = """
WITH event_windows AS (
    SELECT sp.ticker, sp.sector, sp.Date, sp.close_price,
        CASE 
            WHEN sp.Date BETWEEN date('2021-06-04', '-30 days') 
                AND date('2021-06-04', '+30 days') THEN 'wegovy_approval'
            WHEN sp.Date BETWEEN date('2022-05-13', '-30 days') 
                AND date('2022-05-13', '+30 days') THEN 'mounjaro_approval'
            WHEN sp.Date BETWEEN date('2023-11-08', '-30 days') 
                AND date('2023-11-08', '+30 days') THEN 'zepbound_approval'
        END as event_window
    FROM stock_prices sp WHERE ticker != 'SPY'
)
SELECT ticker, sector, event_window,
    COUNT(*) as trading_days,
    ROUND(MIN(close_price), 2) as window_low,
    ROUND(MAX(close_price), 2) as window_high,
    ROUND(MAX(close_price) - MIN(close_price), 2) as price_range,
    ROUND((MAX(close_price) - MIN(close_price)) / MIN(close_price) * 100, 2) as volatility_pct
FROM event_windows
WHERE event_window IS NOT NULL
GROUP BY ticker, sector, event_window
ORDER BY event_window, volatility_pct DESC;
"""
result4 = pd.read_sql_query(query4, conn)
result4.to_csv(os.path.join(processed_path, 'event_volatility.csv'), index=False)
print("\n=== QUERY 4: Event Volatility ===")
print(result4.to_string(index=False))

print("\nAll queries saved to processed folder!")

=== QUERY 1: Price Summary ===
ticker           sector  avg_price  min_price  max_price
   SPY        benchmark     432.38     204.42     757.62
  DXCM diabetes_devices      88.31      27.84     162.82
  PODD diabetes_devices     228.87      72.44     352.82
   MCD        fast_food     234.40     118.61     336.88
   YUM        fast_food     115.68      50.27     167.34
  PLNT          fitness      76.15      27.54     113.55
   KHC    food_beverage      27.42      14.81      35.73
 NSRGY    food_beverage      96.50      65.29     121.04
   PEP    food_beverage     136.42      85.14     175.40
   LLY      glp1_makers     444.73      97.90    1229.93
   NVO      glp1_makers      55.70      19.61     137.40
  ISRG         surgical     322.92     122.58     610.45

=== QUERY 2: Before vs After Wegovy ===
ticker           sector  avg_before_wegovy  avg_after_wegovy  pct_change
   LLY      glp1_makers             133.63            593.68      344.29
   NVO      glp1_makers              26.4

In [11]:
conn.close()
print("Database connection closed")

Database connection closed
